# Expert Intervention Results Visualization

Visualize and compare results from expert routing intervention experiments.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
%matplotlib inline

## Load Results

In [ ]:
# Load summary
results_dir = Path("expert_intervention_results")
with open(results_dir / "summary.json", 'r') as f:
    summary = json.load(f)

print(f"Loaded results for {len(summary)} experiments")
print(f"Experiments: {list(summary.keys())}")

## Compare Refusal Rates

In [ ]:
def plot_refusal_comparison(summary, dataset='jailbreakbench', method='substring_matching'):
    """Plot refusal rates across experiments."""
    
    experiments = []
    refusal_rates = []
    
    for exp_name, exp_data in summary.items():
        if dataset in exp_data and method in exp_data[dataset]:
            experiments.append(exp_name)
            refusal_rates.append(exp_data[dataset][method])
    
    # Sort by refusal rate
    sorted_pairs = sorted(zip(experiments, refusal_rates), key=lambda x: x[1])
    experiments, refusal_rates = zip(*sorted_pairs)
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    colors = ['red' if 'force' in exp else 'blue' if 'suppress' in exp else 'gray' 
              for exp in experiments]
    
    bars = ax.barh(range(len(experiments)), refusal_rates, color=colors, alpha=0.7)
    ax.set_yticks(range(len(experiments)))
    ax.set_yticklabels(experiments)
    ax.set_xlabel('Refusal Rate', fontsize=12, fontweight='bold')
    ax.set_title(f'{dataset} - {method}\nRefusal Rates by Intervention', 
                 fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, (exp, rate) in enumerate(zip(experiments, refusal_rates)):
        ax.text(rate + 0.01, i, f'{rate:.3f}', va='center')
    
    # Add baseline line if present
    if 'baseline' in experiments:
        baseline_idx = experiments.index('baseline')
        baseline_rate = refusal_rates[baseline_idx]
        ax.axvline(baseline_rate, color='green', linestyle='--', linewidth=2, 
                   label='Baseline', alpha=0.5)
        ax.legend()
    
    plt.tight_layout()
    plt.show()

# Plot for jailbreakbench
plot_refusal_comparison(summary, 'jailbreakbench', 'substring_matching')

## Compare Methods

In [ ]:
def plot_method_comparison(summary, dataset='jailbreakbench'):
    """Compare different evaluation methods."""
    
    experiments = []
    substring_rates = []
    llamaguard_rates = []
    
    for exp_name, exp_data in summary.items():
        if dataset in exp_data:
            if 'substring_matching' in exp_data[dataset] and 'llamaguard' in exp_data[dataset]:
                experiments.append(exp_name)
                substring_rates.append(exp_data[dataset]['substring_matching'])
                llamaguard_rates.append(exp_data[dataset]['llamaguard'])
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 8))
    
    x = np.arange(len(experiments))
    width = 0.35
    
    ax.bar(x - width/2, substring_rates, width, label='Substring Matching', alpha=0.8)
    ax.bar(x + width/2, llamaguard_rates, width, label='LlamaGuard', alpha=0.8)
    
    ax.set_xlabel('Experiment', fontsize=12, fontweight='bold')
    ax.set_ylabel('Refusal Rate', fontsize=12, fontweight='bold')
    ax.set_title(f'{dataset}\nRefusal Rates by Evaluation Method', 
                 fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(experiments, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_method_comparison(summary, 'jailbreakbench')

## Effect Size Analysis

In [ ]:
def calculate_effect_sizes(summary, dataset='jailbreakbench', method='substring_matching'):
    """Calculate effect sizes relative to baseline."""
    
    if 'baseline' not in summary or dataset not in summary['baseline']:
        print("Baseline not found!")
        return
    
    baseline_rate = summary['baseline'][dataset][method]
    
    effects = []
    
    for exp_name, exp_data in summary.items():
        if exp_name == 'baseline':
            continue
        
        if dataset in exp_data and method in exp_data[dataset]:
            rate = exp_data[dataset][method]
            effect = rate - baseline_rate
            pct_change = (effect / baseline_rate) * 100
            effects.append((exp_name, effect, pct_change))
    
    # Sort by effect size
    effects.sort(key=lambda x: abs(x[1]), reverse=True)
    
    print(f"Effect Sizes Relative to Baseline ({baseline_rate:.3f})")
    print("=" * 70)
    print(f"{'Experiment':<30} | {'Absolute':<10} | {'Percent'}")
    print("-" * 70)
    
    for exp_name, effect, pct_change in effects:
        print(f"{exp_name:<30} | {effect:+10.3f} | {pct_change:+7.1f}%")
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    exp_names = [e[0] for e in effects]
    effect_sizes = [e[1] for e in effects]
    colors = ['green' if e > 0 else 'red' for e in effect_sizes]
    
    ax.barh(range(len(exp_names)), effect_sizes, color=colors, alpha=0.7)
    ax.set_yticks(range(len(exp_names)))
    ax.set_yticklabels(exp_names)
    ax.axvline(0, color='black', linestyle='-', linewidth=0.8)
    ax.set_xlabel('Effect Size (Change from Baseline)', fontsize=12, fontweight='bold')
    ax.set_title(f'{dataset} - {method}\nEffect Sizes Relative to Baseline',
                 fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

calculate_effect_sizes(summary, 'jailbreakbench', 'substring_matching')

## Cross-Dataset Comparison

In [ ]:
def plot_cross_dataset(summary, method='substring_matching'):
    """Compare interventions across datasets."""
    
    # Get all datasets
    datasets = set()
    for exp_data in summary.values():
        datasets.update(exp_data.keys())
    datasets = sorted(list(datasets))
    
    # Get experiments that have data for all datasets
    valid_experiments = []
    for exp_name, exp_data in summary.items():
        if all(dataset in exp_data and method in exp_data[dataset] 
               for dataset in datasets):
            valid_experiments.append(exp_name)
    
    if not valid_experiments:
        print("No experiments with data for all datasets")
        return
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 8))
    
    x = np.arange(len(valid_experiments))
    width = 0.8 / len(datasets)
    
    for i, dataset in enumerate(datasets):
        rates = [summary[exp][dataset][method] for exp in valid_experiments]
        offset = (i - len(datasets)/2 + 0.5) * width
        ax.bar(x + offset, rates, width, label=dataset, alpha=0.8)
    
    ax.set_xlabel('Experiment', fontsize=12, fontweight='bold')
    ax.set_ylabel('Refusal Rate', fontsize=12, fontweight='bold')
    ax.set_title(f'{method}\nCross-Dataset Comparison',
                 fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(valid_experiments, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_cross_dataset(summary, 'substring_matching')

## Sample Completions Analysis

In [ ]:
def load_completions(results_dir, experiment, dataset):
    """Load completions for an experiment."""
    path = results_dir / experiment / f"{dataset}_completions.json"
    if not path.exists():
        return None
    with open(path, 'r') as f:
        return json.load(f)

def compare_completions(results_dir, experiments, dataset, sample_idx=0):
    """Compare completions for the same prompt across experiments."""
    
    print(f"Comparing completions for {dataset}, sample {sample_idx}")
    print("=" * 80)
    
    for exp_name in experiments:
        completions = load_completions(results_dir, exp_name, dataset)
        if completions and sample_idx < len(completions):
            sample = completions[sample_idx]
            print(f"\n{exp_name.upper()}:")
            print(f"Instruction: {sample['instruction'][:100]}...")
            print(f"Completion: {sample['completion'][:200]}...")
            print("-" * 80)

# Compare first sample across key experiments
compare_completions(
    results_dir,
    ['baseline', 'suppress_harmful', 'force_harmful'],
    'jailbreakbench',
    sample_idx=0
)